In [15]:
import os
import h5py
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path
from utils import data_loader_utils

### Initial data visualization / validation of format

In [16]:
# Path for main dataset data folder
data_path = Path("./data").absolute()
# List of all h5 files in the data folder
file_list = data_loader_utils.find_all_h5s_in_dir(data_path)

In [17]:
# Test - Reading and formatting first file
file_df = data_loader_utils.format_file(file_list[0])
file_df

,X,Y,Z,timestamp,label,machine,process,i,window
0,-15.0,25.0,-1013.0,0.0,True,1,2,13,0
1,-21.0,19.0,-1013.0,0.5,True,1,2,13,0
2,-17.0,29.0,-1013.0,1.0,True,1,2,13,0
3,-13.0,13.0,-1013.0,1.5,True,1,2,13,0
4,-9.0,25.0,-1020.0,2.0,True,1,2,13,0
...,...,...,...,...,...,...,...,...,...
94203,0.0,23.0,-1013.0,47101.5,True,1,2,13,23
94204,-19.0,11.0,-1018.0,47102.0,True,1,2,13,23
94205,-1.0,31.0,-1015.0,47102.5,True,1,2,13,23
94206,-19.0,9.0,-1020.0,47103.0,True,1,2,13,23


### Splitting training and testing buckets

In [35]:
good_file_list = [f for f in file_list if "good" in f]
bad_file_list = [f for f in file_list if "bad" in f]

In [44]:
X = np.array(good_file_list)
F = np.array(bad_file_list)
# Number of files per label
healthy = len(good_file_list)
total = len(file_list)
faulty = total - healthy
# Threshold definition
np.random.seed(42)
shuffled_indices = np.random.permutation(len(X))
test_ratio_h = 0.20 - (faulty / total)
test_set_size = int(len(X) * test_ratio_h)
# Slice indices and partition data
train_indices = shuffled_indices[test_set_size:]
test_indices = shuffled_indices[:test_set_size]

X_train, X_test = X[train_indices], np.concatenate((X[test_indices],F))
print(len(X_train), len(X_test))

1373 329


### Final data preparation for model training
Training data must be converted from Dataframe format to Numpy array format

In [51]:
for i in range(0, 1):
    df = file_df[file_df.window == i]
    A = df[["X","Y","Z"]].to_numpy()
    
A

array([[  -15.,    25., -1013.],
       [  -21.,    19., -1013.],
       [  -17.,    29., -1013.],
       ...,
       [   27.,   -81., -1110.],
       [   21.,   -50., -1128.],
       [    0.,    54., -1100.]], shape=(4000, 3))